# Day 53: Calling an LLM API

**Today's question:** how does our program talk to a real model on someone else's computer, safely and without
wasting money?

Everything in this notebook works without an API key (we then use the mock model from `llm.py`).

In [ ]:
import os
from pathlib import Path

import anthropic
from dotenv import load_dotenv

from llm import mock_llm

## 1. What an API is

An API is like a waiter: we send a **request** in an agreed format, a program on Anthropic's servers runs the model,
and we get a **response** back. A request is data we already know how to build:

In [ ]:
request = {
    "model": "claude-opus-5-5",
    "max_tokens": 16000,
    "system": "You label customer reviews for an online shop.",
    "messages": [{"role": "user", "content": "Classify this review in one word: Great lamp!"}],
}
for key, value in request.items():
    print(f"{key:<11} {value}")

The response contains:
- `content`: a **list of blocks**. Text blocks hold the answer; a *thinking* block may come first.
- `stop_reason`: why the model stopped, e.g. `"end_turn"` (finished) or `"refusal"` (declined).
- `usage`: how many input and output tokens were used, which is what we pay for.

Every request carries an **API key**: a secret that says "this is my account, bill me".

## 2. Cost and rate limits

Prices are per **million tokens**, different for input and output (October 2026 list prices; always check the
current pricing page).

In [ ]:
PRICES = {  # model -> (input $ per 1M tokens, output $ per 1M tokens)
    "claude-opus-5-5": (4.00, 20.00),
    "claude-haiku-5-5": (0.10, 0.50),
}


def estimate_cost(model, input_tokens, output_tokens):
    """Cost in dollars of a job, using the PRICES table."""
    input_price, output_price = PRICES[model]
    # 1_000_000 is one million: Python allows _ in numbers to make them easier to read
    return input_tokens / 1_000_000 * input_price + output_tokens / 1_000_000 * output_price


# 40 reviews: about 120 tokens of prompt and 60 tokens of reply each
for model in PRICES:
    print(f"{model:<17} ${estimate_cost(model, 40 * 120, 40 * 60):.4f}")

🤔 **Predict:** how much would **100,000** reviews cost with each model?

In [ ]:
for model in PRICES:
    print(f"{model:<17} ${estimate_cost(model, 100_000 * 120, 100_000 * 60):,.2f}")

- Output tokens cost 5× more than input: ask for one word or compact JSON, not essays.
- The model may also *think* before answering; thinking counts as output tokens. Our numbers are a **minimum**.
- **Rate limits:** an account may only send so many requests per minute. Too many → error 429
  (`anthropic.RateLimitError`). The SDK automatically waits and retries **twice** first.

## 3. Environment variables

Every program receives a set of text settings from the operating system: `os.environ`. It works like a dict.

In [ ]:
os.environ["SHOP_NAME"] = "Mei's Gadgets"   # set for this running program only
print(os.environ.get("SHOP_NAME"))
print(os.environ.get("SHOP_CITY"))
print(os.environ.get("SHOP_CITY", "Lahore"))

In [ ]:
os.environ["SHOP_CITY"]   # like a dict: a missing key is a KeyError

**Never print a key.** Print whether it is set:

In [ ]:
key_is_set = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("API key available:", key_is_set)

## 4. `.env` files and `.gitignore`

A `.env` file holds `NAME=value` lines. `load_dotenv()` copies them into `os.environ`. Here is a harmless demo
file (no secrets) that we create and load:

In [ ]:
Path("shop_settings.env").write_text("SHOP_CITY=Karachi\nSHOP_CURRENCY=PKR\n", encoding="utf-8")

found = load_dotenv("shop_settings.env")
print("file found:", found)
print(os.environ.get("SHOP_CITY"), os.environ.get("SHOP_CURRENCY"))

For the real key, the course root has `.env.example`. You copy it to `.env` and fill in your values.
`load_dotenv()` (no file name) searches the current folder and then each parent folder for `.env`.

In [ ]:
print(Path("../../../.env.example").read_text(encoding="utf-8"))

`.gitignore` lists files that Git must never upload to GitHub. Is `.env` protected?

In [ ]:
ignored = Path("../../../.gitignore").read_text(encoding="utf-8").splitlines()
print(".env" in ignored)

**Order matters.** `llm.py` starts like this:

```python
load_dotenv()                                          # 1. copy .env into os.environ
MODEL = os.environ.get("LLM_MODEL", "claude-opus-5-5") # 2. then read the setting
```

🤔 **Predict:** what goes wrong if these two lines are swapped and `.env` says `LLM_MODEL=claude-haiku-5-5`?

## 5. Building `ask_llm()` step by step

**Step 1: offline only.** Everything we needed on Days 51–52.

In [ ]:
def ask_llm_v1(prompt):
    return mock_llm(prompt)


print(ask_llm_v1("Classify this review in one word: <review>The zip broke.</review>"))

**Step 2: choose.** Use the mock when forced offline (`COURSE_OFFLINE` is `"1"`, as on the course checker) or when
there is no key.

In [ ]:
def ask_llm_v2(prompt):
    if os.environ.get("COURSE_OFFLINE") == "1" or not os.environ.get("ANTHROPIC_API_KEY"):
        return mock_llm(prompt)
    return "(step 3 will call the real model here)"


print(ask_llm_v2("Classify this review in one word: <review>The zip broke.</review>"))

**Step 3: the real call.** This is exactly the function in `llm.py`.

In [ ]:
MODEL = os.environ.get("LLM_MODEL", "claude-opus-5-5")


def ask_llm(prompt, system="You are a helpful assistant."):
    """Send one prompt to Claude and return its text reply (mock reply when offline)."""
    if os.environ.get("COURSE_OFFLINE") == "1" or not os.environ.get("ANTHROPIC_API_KEY"):
        return mock_llm(prompt)
    client = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY automatically
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=16000,
        system=system,
        messages=[{"role": "user", "content": prompt}],
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",  # if this model declines, the API retries on a suitable fallback model
    )
    if response.stop_reason == "refusal":
        return "[The model declined this request.]"
    # The reply is a list of blocks (thinking blocks may come first), so keep only the text blocks.
    return "".join([block.text for block in response.content if block.type == "text"])


print(ask_llm("Classify this review in one word: <review>The zip broke.</review>"))

| Line | Why |
|------|-----|
| `anthropic.Anthropic()` | the client reads `ANTHROPIC_API_KEY` from `os.environ` itself: no key in the code |
| `model=MODEL` | exact model ID string, chosen in `.env` |
| `max_tokens=16000` | a safety limit for the reply length, not a target |
| `betas=[...]`, `fallbacks="default"` | if the model declines, the API retries on a suitable fallback model (a beta feature, so we name it) |
| `stop_reason == "refusal"` | declined: return a clear message instead of crashing |
| `"".join([... if block.type == "text"])` | keep only text blocks |

## 6. Why only the text blocks? (fake responses, offline)

We cannot call the API offline, but we can build **fake response objects** with the same shape (Week 7 classes).

In [ ]:
class FakeTextBlock:
    def __init__(self, text):
        self.type = "text"
        self.text = text


class FakeThinkingBlock:
    def __init__(self):
        self.type = "thinking"
        self.thinking = ""


class FakeResponse:
    def __init__(self, content, stop_reason="end_turn"):
        self.content = content
        self.stop_reason = stop_reason


response = FakeResponse([FakeThinkingBlock(), FakeTextBlock("negative")])

In [ ]:
response.content[0].text   # the first block is a thinking block: it has no .text!

In [ ]:
def reply_text(response):
    """The last lines of ask_llm, as a function we can test offline."""
    if response.stop_reason == "refusal":
        return "[The model declined this request.]"
    return "".join([block.text for block in response.content if block.type == "text"])


print(reply_text(response))
print(reply_text(FakeResponse([FakeTextBlock("Hello "), FakeTextBlock("there")])))
print(reply_text(FakeResponse([], stop_reason="refusal")))

## 7. When the API call fails

Some anthropic errors are *kinds of* other errors, just like on Day 25:

In [ ]:
print(issubclass(anthropic.RateLimitError, anthropic.APIStatusError))
print(issubclass(anthropic.AuthenticationError, anthropic.APIStatusError))
print(issubclass(anthropic.APIConnectionError, anthropic.APIStatusError))

So the **most specific** `except` must come first. `safe_ask` returns `None` on failure so the caller can count it.

In [ ]:
def safe_ask(prompt):
    """ask_llm, but an API problem prints a clear message and returns None instead of crashing."""
    try:
        return ask_llm(prompt)
    except anthropic.AuthenticationError:
        print("The API key was rejected. Check ANTHROPIC_API_KEY in your .env file.")
    except anthropic.RateLimitError:
        print("Rate limit reached (even after automatic retries). Wait a minute and try again.")
    except anthropic.APIConnectionError:
        print("Could not reach the API. Check your internet connection.")
    except anthropic.APIStatusError as error:
        print("The API returned an error with status", error.status_code)
    return None


print(safe_ask("Summarise this review: <review>Bright, adjustable and easy on the eyes.</review>"))

🤔 **Predict:** if `except anthropic.APIStatusError` were first, which messages could never be printed?

## 8. Using the finished helper

The same code lives in `llm.py`. In every later notebook and project we just import it:

In [ ]:
import llm

print("Model setting:", llm.MODEL)
for review in ["Love it! Easy to clean.", "Arrived late and the jug was cracked."]:
    print(llm.ask_llm(f"Classify this review in one word: <review>{review}</review>"))

## Recap

1. An API key identifies and bills your account. It lives in `.env` (listed in `.gitignore`), never in code. Never
   print it.
2. Cost = input tokens × input price + output tokens × output price, per million. Rate limits → 429 → the SDK
   retries twice.
3. `os.environ.get(name, default)` reads settings; `load_dotenv()` loads `.env` first.
4. `ask_llm()`: mock when offline or without a key; otherwise call Claude, check for a refusal, join only the text
   blocks.
5. Catch API errors most specific first.

**Next time:** asking for JSON and validating every field of the reply.